# Phase 2: tokenizer + token shards (CPU session, ~2-4 h)
Run once. Accept the terms of `bigcode/the-stack-dedup` on huggingface.co first (it is gated).
Output: a private HF dataset `$HF_USERNAME/minillm-tokens` with uint16 shards + the tokenizer.

Each source is uploaded as soon as it finishes. If the session hits the 12 h cap, just run the notebook again:
it reuses the uploaded tokenizer and skips sources that are already on the Hub.

In [ ]:
# Kaggle: Add-ons -> Secrets: HF_TOKEN (write), HF_USERNAME, WANDB_API_KEY. Settings -> Internet ON.
import os
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
for name in ["HF_TOKEN", "HF_USERNAME", "WANDB_API_KEY"]:
    try:
        os.environ[name] = secrets.get_secret(name)
    except Exception:
        print(f"secret {name} not set")

GITHUB_REPO = "https://github.com/Dhagash1206/MiniLLM.git"
%cd /kaggle/working
!git clone -q {GITHUB_REPO} minillm 2>/dev/null || (cd minillm && git pull -q)
%cd /kaggle/working/minillm
!pip install -q -e .

## Train the 32k byte-level BPE tokenizer (or reuse the one already on the Hub)

In [ ]:
!python -m MiniLLM.Data.Tokenizer --out artifacts/tokenizer --docs-per-source 150000 \
    --reuse-from-hub $HF_USERNAME/minillm-tokens

## Filter, dedup, tokenize, upload
Source parquet files are downloaded whole (in parallel) to `/kaggle/temp`, which has more disk than `/kaggle/working`,
and deleted once read.

In [ ]:
!python -m MiniLLM.Data.PrepareShards --tokenizer artifacts/tokenizer --out /kaggle/temp/tokens \
    --download-dir /kaggle/temp/hf_files --total-tokens 2.6e9 --push-to-hub $HF_USERNAME/minillm-tokens